# 03 -- Phase 2: Integration & Astrometry

Group calibrated frames, align them (`astroalign`), inverse-variance stack
them, and plate-solve the master with Astrometry.net. The ERR plane is warped
and combined alongside the data; the master gets a WCS.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG  --  EDIT THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================
import os

# 1) Shared, read-only Astrometry.net index directory (used by Phase 2):
os.environ["CASSA_ASTROMETRY_INDEX"] = os.path.abspath("../../astrometry_data")

# 2) The provided workshop dataset + a writable work directory:
RAW_DIR   = "../raw"     # provided raw frames, one level up from notebooks/
WORK_DIR  = "../work"    # writable output dir, one level up from notebooks/

RAW_DIR   = os.path.abspath(os.path.expandvars(RAW_DIR))
WORK_DIR  = os.path.abspath(os.path.expandvars(WORK_DIR))

# These are relative to the working directory, which Jupyter sets to this
# notebook's folder. Fail loudly here rather than confusingly further down.
assert os.path.isdir(RAW_DIR), (
    f"RAW_DIR not found: {RAW_DIR}\nRun this notebook from the notebooks/ "
    f"directory, or set RAW_DIR/WORK_DIR to absolute paths above."
)

# Each phase writes into its own directory under WORK_DIR.
PHASE1_DIR = os.path.join(WORK_DIR, "phase1")   # calibrated frames
PHASE2_DIR = os.path.join(WORK_DIR, "phase2")   # master stacks + WCS
PHASE3_DIR = os.path.join(WORK_DIR, "phase3")   # flux-calibrated + catalogs
PHASE4_DIR = os.path.join(WORK_DIR, "phase4")   # diagnostics report
os.makedirs(WORK_DIR, exist_ok=True)
print("RAW_DIR    =", RAW_DIR)
print("PHASE1_DIR =", PHASE1_DIR)
print("PHASE2_DIR =", PHASE2_DIR)

## Run Phase 2
Uses the shared **local** astrometry index directory set in the config cell
(`CASSA_ASTROMETRY_INDEX`) -- plate solving needs no network. Reads `PHASE1_DIR`
and writes the master stacks into `PHASE2_DIR`.

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase2_integration.pipeline import IntegrationPipeline
cfg = load_config()
pipe = IntegrationPipeline(PHASE1_DIR, output_dir=PHASE2_DIR, config=cfg)
pipe.setup()
pipe.execute()
run_dir = pipe.run_dir
print('Phase 2 output:', run_dir)

## Inspect the master stack + its WCS

In [ ]:
import glob, os, numpy as np
from astropy.wcs import WCS
from cassa_photometry.fits_utils import read_mef
masters = sorted(glob.glob(os.path.join(run_dir, 'Master_*.fits')))
sci, err, dq, hdr = read_mef(masters[0])
print('STACKCNT:', hdr.get('STACKCNT'), ' TOT_EXP:', hdr.get('TOT_EXP'))
w = WCS(hdr)
print('WCS celestial:', w.has_celestial)
ny, nx = sci.shape
print('Field centre:', w.pixel_to_world(nx/2, ny/2).to_string('hmsdms'))

### Exercise 1 -- how much deeper is the stack?
The stack of $N$ frames should be roughly $\sqrt{N}$ deeper than one frame.
Compare the median ERR of the master to that of a single calibrated frame in
the same filter.

In [ ]:
import glob, os
import numpy as np
from cassa_photometry.fits_utils import read_mef

N = hdr.get('STACKCNT')
filt = hdr.get('FILTER')

# Find a single calibrated frame in the same filter as the master, for a fair comparison.
single_path = None
for f in sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits'))):
    _, _, _, h = read_mef(f)
    if h.get('FILTER') == filt:
        single_path = f
        break

_, single_err, _, _ = read_mef(single_path)
med_err_single = float(np.nanmedian(single_err))
med_err_master = float(np.nanmedian(err))

print(f"Single frame : {os.path.basename(single_path)}")
print(f"Master       : {os.path.basename(masters[0])}  (N={N} frames, filter={filt})\n")
print(f"Median ERR (single) : {med_err_single:.3f} e-")
print(f"Median ERR (master) : {med_err_master:.3f} e-")
print(f"Observed depth boost (single/master): {med_err_single / med_err_master:.3f}")
print(f"Expected sqrt(N)                    : {np.sqrt(N):.3f}")

### Exercise 2 -- what did the plate solve buy us?
Before Phase 2 the frames had pixels; now they have sky coordinates. Read the
plate scale straight out of the WCS, work out the field of view, and check
that a pixel -> sky -> pixel round trip returns where it started. Compare the
measured scale against the `SECPIX` keyword the camera wrote -- do they agree?

In [ ]:
from astropy.wcs.utils import proj_plane_pixel_scales

# The WCS is the authority on the plate scale -- read it from the solution itself.
scales_deg = proj_plane_pixel_scales(w.celestial)
scale_arcsec = scales_deg * 3600.0
print(f"Plate scale from the WCS : {scale_arcsec[0]:.4f} x {scale_arcsec[1]:.4f} arcsec/pixel")
print(f"PIXSCALE header keyword  : {hdr.get('PIXSCALE')}   <- never filled in by the pipeline")
print(f"SECPIX (from the camera) : {hdr.get('SECPIX')} arcsec/pixel")
print("\nNote the disagreement: SECPIX is the value the acquisition software wrote,")
print("while the WCS scale was *measured* by matching real stars against the sky.")
print("When they differ, the astrometric solution is the one to trust -- and a large")
print("gap like this is worth chasing down (binning? wrong nominal focal length?).\n")

fov_x = nx * scale_arcsec[0] / 60.0
fov_y = ny * scale_arcsec[1] / 60.0
print(f"Image size    : {nx} x {ny} pixels")
print(f"Field of view : {fov_x:.2f} x {fov_y:.2f} arcmin")

# Pixel -> sky -> pixel round trip: does the solution invert cleanly?
test_pix = [(0, 0), (nx / 2, ny / 2), (nx - 1, ny - 1)]
print(f"\n{'pixel':>18} {'-> sky (RA, Dec deg)':>32} {'-> back to pixel':>26}")
for px, py in test_pix:
    sky = w.pixel_to_world(px, py)
    bx, by = w.world_to_pixel(sky)
    print(f"{f'({px:.1f}, {py:.1f})':>18} {f'({sky.ra.deg:.5f}, {sky.dec.deg:.5f})':>32} "
          f"{f'({bx:.3f}, {by:.3f})':>26}")

# How far apart are the two corners on the sky?
c0 = w.pixel_to_world(0, 0)
c1 = w.pixel_to_world(nx - 1, ny - 1)
print(f"\nCorner-to-corner separation: {c0.separation(c1).arcmin:.2f} arcmin (the diagonal)")
print(f"Target from the header     : {hdr.get('OBJECT')}")


### Exercise 3 -- does inverse-variance weighting do what the theory says?
The lecture claims $\mathrm{var}_{\rm master} = 1/\sum_i (1/\mathrm{var}_i)$.
Collect the median ERR of every calibrated frame that went into this master,
predict the stacked error from that formula, and compare it with what the
master actually has. Also compute the naive "mean error / $\sqrt{N}$" and see
when the two predictions would part company.

In [ ]:
# Theory (lecture): stacking with weights w_i = 1/var_i gives var_master = 1 / sum(1/var_i).
# Test it against the frames that actually went into this master.
singles = []
for f in sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits'))):
    _, e, _, h = read_mef(f)
    if h.get('FILTER') == filt:
        singles.append((os.path.basename(f), float(np.nanmedian(e))))

print(f"Frames in filter {filt} (master says STACKCNT = {N}):\n")
print(f"{'frame':>52} {'median ERR (e-)':>16}")
for name, med in singles:
    print(f"{name:>52} {med:16.3f}")

err_i = np.array([m for _, m in singles])
var_i = err_i ** 2

predicted_var = 1.0 / np.sum(1.0 / var_i)      # inverse-variance combination
predicted_err = np.sqrt(predicted_var)
equal_weight_err = np.mean(err_i) / np.sqrt(len(err_i))   # the naive sqrt(N) rule

print(f"\n{'Predicted ERR, inverse-variance 1/sqrt(sum 1/var_i)':>52} : {predicted_err:7.3f} e-")
print(f"{'Predicted ERR, naive mean/sqrt(N)':>52} : {equal_weight_err:7.3f} e-")
print(f"{'Actual median ERR of the master':>52} : {med_err_master:7.3f} e-")
print(f"\nActual / inverse-variance prediction: {med_err_master / predicted_err:.3f}")

print("\nThe frames here have very similar noise, so both predictions nearly agree.")
print("Inverse-variance weighting only pulls ahead when the frames differ -- a")
print("cloudy or high-airmass frame gets down-weighted instead of dragging the")
print("stack down. The small residual gap comes from resampling during warping,")
print("which correlates neighbouring pixels slightly.")
